# Fine-tuning NepConformer on Nwāchā Munā: Nepal Bhasha (Newari) ASR

**Research:** *Nwāchā Munā — Proximal Cross-Lingual Transfer for Ultra-Low-Resource Nepal Bhasha ASR*

This notebook fine-tunes a **Nepali Conformer-CTC-BPE** model on **5.39 hours** of manually transcribed Devanagari speech in Nepal Bhasha (Newari). The core hypothesis is that *proximal cross-lingual transfer* from Nepali — a geographically and linguistically adjacent language — can rival large-scale multilingual pretraining (e.g., Whisper-Small) in an ultra-low-resource ASR setting.

---

## Notebook Overview

| Step | Description |
|------|-------------|
| 1 | Install dependencies (WandB, NeMo, FastText) |
| 2 | Configure experiment tracking (WandB) |
| 3 | Clone NVIDIA NeMo repository |
| 4 | Convert stereo audio → mono (NeMo requirement) |
| 5 | Define Conformer-CTC-BPE training config |
| 6 | Patch NeMo training script (NumPy 2.0 + PyTorch 2.6 compatibility) |
| 7 | Launch fine-tuning |

---

## Environment
- **GPU:** NVIDIA Tesla T4 (Kaggle)
- **Framework:** NVIDIA NeMo 2.1.0
- **Model:** Conformer-CTC-BPE (18-layer, initialized from Nepali checkpoint)
- **Dataset:** Nwāchā Munā — 5.39 hrs Newari speech, Devanagari transcriptions


---
## Step 1 — Install Dependencies

We need three external packages beyond the default Kaggle environment:
- **`wandb`** — experiment tracking and metric logging
- **`pybind11`** — C++ binding library required by NeMo's text processing modules
- **`fasttext`** — language identification utility used internally by NeMo
- **`nemo_toolkit[asr]`** — NVIDIA's end-to-end ASR framework (pinned to 2.1.0 for reproducibility)

In [ ]:
!pip install wandb
!pip install pybind11
!pip install fasttext
!pip install "nemo_toolkit[asr]==2.1.0"

---
## Step 2 — Configure Experiment Tracking (Weights & Biases)

We store the WandB API key in a local file (`api.txt`) and load it into the environment. This avoids hardcoding credentials in the notebook.

> **Note:** Replace the key string below with your own WandB API key before running.

In [ ]:
import os

# Write the WandB API key to a local file
# Replace the key below with your own key from https://wandb.ai/authorize
WANDB_API_KEY = "your wandb api key"  # <-- replace with your key

with open("api.txt", "w") as f:
    f.write(WANDB_API_KEY)

# Load the key into the environment so NeMo's WandB logger picks it up automatically
with open("api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()

print("WandB API key loaded into environment.")

---
## Step 3 — Clone NVIDIA NeMo Repository

We clone NeMo directly from GitHub to access its training scripts and utilities. NeMo provides the `speech_to_text_ctc_bpe.py` entry point we will use for fine-tuning.

In [ ]:
!git clone https://github.com/NVIDIA/NeMo.git

---
## Step 4 — Convert Audio from Stereo to Mono

NeMo's ASR pipeline expects **single-channel (mono) audio**. Some recordings in the Nwāchā Munā corpus may be stereo. This step:

1. Reads each manifest file (`nemo_train.json`, `nemo_val.json`, `nemo_test.json`)
2. Loads the referenced audio file
3. If stereo, converts to mono by averaging channels
4. Saves the processed audio to `/kaggle/working/mono_audio/`
5. Writes updated manifests (prefixed with `mono_`) pointing to the new audio paths

**Manifest format (NeMo JSONL):**
```json
{"audio_filepath": "/path/to/audio.wav", "duration": 3.5, "text": "नेपाल भाषा"}
```

In [ ]:
import json
import os
import soundfile as sf
import numpy as np
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_DIR  = "/kaggle/input/final-data"       # original dataset location
OUTPUT_DIR = "/kaggle/working/mono_audio"     # where mono audio files are saved
WORKING_DIR = "/kaggle/working"               # where updated manifests are written

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Manifests to process
MANIFEST_FILES = ["nemo_train.json", "nemo_val.json", "nemo_test.json"]


def convert_to_mono(manifest_filename):
    """Read a NeMo manifest, convert referenced audio to mono, write an updated manifest."""
    input_path = os.path.join(INPUT_DIR, manifest_filename)
    output_manifest_path = os.path.join(WORKING_DIR, f"mono_{manifest_filename}")

    if not os.path.exists(input_path):
        print(f"[SKIP] {manifest_filename} not found at {input_path}")
        return

    updated_entries = []
    print(f"\n[Processing] {manifest_filename}")

    with open(input_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="  Converting audio"):
            entry = json.loads(line)
            original_path = entry["audio_filepath"]

            # Resolve path: try absolute first, then relative to INPUT_DIR
            full_path = (
                original_path
                if os.path.exists(original_path)
                else os.path.join(INPUT_DIR, original_path)
            )

            new_audio_path = os.path.join(OUTPUT_DIR, os.path.basename(full_path))

            try:
                audio, sample_rate = sf.read(full_path)

                # Convert stereo → mono by averaging left and right channels
                if audio.ndim > 1 and audio.shape[1] > 1:
                    audio = audio.mean(axis=1)

                sf.write(new_audio_path, audio, sample_rate)
                entry["audio_filepath"] = new_audio_path
                updated_entries.append(entry)

            except Exception as e:
                print(f"  [ERROR] {os.path.basename(full_path)}: {e}")

    # Write the updated manifest
    with open(output_manifest_path, "w", encoding="utf-8") as f_out:
        for entry in updated_entries:
            f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"  [Done] Saved updated manifest → {output_manifest_path}")


# Run conversion for all splits
for manifest in MANIFEST_FILES:
    convert_to_mono(manifest)

print("\nAll audio files converted to mono. Manifests updated.")

---
## Step 5 — Define Training Configuration (Conformer-CTC-BPE)

We write a YAML config file consumed by NeMo's Hydra-based training script. Key design choices:

| Config Section | Choice | Rationale |
|---|---|---|
| **Tokenizer** | Pre-trained Newari BPE vocabulary | Reuse vocab from initial Newari data; avoids retraining tokenizer |
| **Encoder** | 18-layer Conformer, d_model=256 | Matches the Nepali source checkpoint architecture |
| **Optimizer** | AdamW, lr=1e-4, CosineAnnealing | Standard for Conformer fine-tuning |
| **Spec Augment** | 2 freq masks (width 27), 2 time masks (width 70) | Switchboard-strong policy for robustness on small data |
| **Accumulate grad batches** | 8 | Effective batch size = 4 × 8 = 32, compensating for small physical batch |
| **Max epochs** | 60 | Early stopping disabled; best checkpoint selected by `val_wer` |



In [ ]:
# ── Training Configuration ─────────────────────────────────────────────────────
# This YAML is consumed by NeMo's Hydra runner.
# The model is initialized from a Nepali Conformer checkpoint via the CLI flag
# ++init_from_ptl_ckpt (see Step 7), NOT from init_from_ptl_ckpt in this config.

config = """
name: "Conformer-CTC-BPE-Finetune-Newari"

# Checkpoint init is handled via CLI: ++init_from_ptl_ckpt=<path>
init_from_ptl_ckpt: null

model:
  sample_rate: 16000
  log_prediction: true       # log sample predictions to WandB each validation
  ctc_reduction: 'mean_batch'
  skip_nan_grad: false
  use_cer: true              # report Character Error Rate (CER) alongside WER

  # ── Data ────────────────────────────────────────────────────────────────────
  train_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_train.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7       # discard utterances longer than 16.7s (memory limit)
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: "synced_randomized"  # group similar-length audio for efficiency
    bucketing_batch_size: null

  validation_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_val.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  test_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_test.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  # ── Tokenizer ───────────────────────────────────────────────────────────────
  # Using a pre-trained Newari BPE vocabulary (Devanagari script)
  tokenizer:
    dir: "/kaggle/input/datasets/jennypoudel/newari-data/"
    type: bpe

  # ── Feature Extraction ──────────────────────────────────────────────────────
  preprocessor:
    _target_: nemo.collections.asr.modules.AudioToMelSpectrogramPreprocessor
    sample_rate: ${model.sample_rate}
    normalize: "per_feature"   # per-feature mean/variance normalization
    window_size: 0.025         # 25ms frames
    window_stride: 0.01        # 10ms hop
    window: "hann"
    features: 80               # 80-dim log-mel filterbank
    n_fft: 512
    log: true
    frame_splicing: 1
    dither: 0.00001            # small dither for numerical stability
    pad_to: 0
    pad_value: 0.0

  # ── SpecAugment (Switchboard-Strong Policy) ──────────────────────────────────
  spec_augment:
    _target_: nemo.collections.asr.modules.SpectrogramAugmentation
    freq_masks: 2
    time_masks: 2
    freq_width: 27
    time_width: 70

  # ── Encoder (Conformer-Medium) ───────────────────────────────────────────────
  encoder:
    _target_: nemo.collections.asr.modules.ConformerEncoder
    feat_in: ${model.preprocessor.features}  # 80
    feat_out: -1                              # determined by d_model
    n_layers: 18
    d_model: 256
    subsampling: striding
    subsampling_factor: 4                     # 4x time-dimension reduction
    subsampling_conv_channels: -1
    causal_downsampling: false
    ff_expansion_factor: 4
    self_attention_model: rel_pos             # relative positional encoding
    n_heads: 4
    att_context_size: [-1, -1]               # full context (non-streaming)
    att_context_style: regular
    xscaling: true
    untie_biases: true
    pos_emb_max_len: 5000
    conv_kernel_size: 31
    conv_norm_type: 'batch_norm'
    conv_context_size: null
    dropout: 0.1
    dropout_pre_encoder: 0.1
    dropout_emb: 0.0
    dropout_att: 0.1
    stochastic_depth_drop_prob: 0.0
    stochastic_depth_mode: linear
    stochastic_depth_start_layer: 1

  # ── Decoder (CTC linear projection) ─────────────────────────────────────────
  decoder:
    _target_: nemo.collections.asr.modules.ConvASRDecoder
    feat_in: null
    num_classes: -1   # inferred from tokenizer vocabulary size
    vocabulary: []

  interctc:
    loss_weights: []
    apply_at_layers: []

  # ── Optimizer ───────────────────────────────────────────────────────────────
  optim:
    name: adamw
    lr: 0.0001
    betas: [0.9, 0.98]
    weight_decay: 1e-3
    sched:
      name: CosineAnnealing
      min_lr: 1e-6
      warmup_steps: 3000    # ~linear warmup for first 3000 optimizer steps

# ── Trainer ──────────────────────────────────────────────────────────────────
trainer:
  devices: -1              # use all available GPUs
  num_nodes: 1
  max_epochs: 100
  max_steps: -1
  val_check_interval: 1.0  # validate every epoch
  accelerator: auto
  accumulate_grad_batches: 8   # effective batch = 4 * 8 = 32
  gradient_clip_val: 1.0
  precision: 32
  log_every_n_steps: 100
  enable_progress_bar: True
  num_sanity_val_steps: 0
  check_val_every_n_epoch: 1
  sync_batchnorm: true
  enable_checkpointing: False  # managed by exp_manager below
  logger: false
  benchmark: false
  max_time: "00:11:30:00"     # stop at 11h30m (Kaggle session limit)

# ── Experiment Manager ────────────────────────────────────────────────────────
exp_manager:
  exp_dir: null
  name: ${name}
  create_tensorboard_logger: true
  create_checkpoint_callback: true
  checkpoint_callback_params:
    monitor: "val_wer"    # save top-3 checkpoints by validation WER
    mode: "min"
    save_top_k: 3
    always_save_nemo: True
  create_early_stopping_callback: False  # disabled: let training run full 100 epochs
  resume_if_exists: false
  resume_ignore_no_checkpoint: false
  create_wandb_logger: True
  wandb_logger_kwargs:
    name: "finetune-newari-asr-semi-supervised"
    project: "Conformer-Finetuning"
"""

config_path = "finetune_config.yaml"
with open(config_path, "w") as f:
    f.write(config)

print(f"Training config written to: {config_path}")


## Step 6 — Patch NeMo Training Script for Compatibility


In [ ]:
NEMO_SCRIPT_PATH = "NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py"

patched_script = '''
# Copyright (c) 2020, NVIDIA CORPORATION.  All rights reserved.
# Licensed under the Apache License, Version 2.0.
# See https://www.apache.org/licenses/LICENSE-2.0 for the full license.

import lightning.pytorch as pl
from omegaconf import OmegaConf
import torch
import numpy as np

# ── Compatibility Fix 1: NumPy 2.0 removed np.sctypes ────────────────────────
# NeMo < 2.0 internally uses np.sctypes which was removed in NumPy 2.0.
# We restore a compatible mapping so NeMo can import without errors.
if not hasattr(np, "sctypes"):
    np.sctypes = {
        "int":     [np.int8, np.int16, np.int32, np.int64],
        "uint":    [np.uint8, np.uint16, np.uint32, np.uint64],
        "float":   [np.float16, np.float32, np.float64],
        "complex": [np.complex64, np.complex128],
        "others":  [bool, object, bytes, str, np.void],
    }

# ── Compatibility Fix 2: PyTorch 2.6+ weights_only default ───────────────────
# PyTorch 2.6 changed torch.load default to weights_only=True, which breaks
# NeMo checkpoint loading. We patch torch.load to restore the old behaviour.
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load
# ─────────────────────────────────────────────────────────────────────────────

import importlib.util

from nemo.collections.asr.models.ctc_bpe_models import EncDecCTCModelBPE
from nemo.core.config import hydra_runner
from nemo.utils import logging
from nemo.utils.exp_manager import exp_manager

# Load resolve_trainer_cfg directly from file (avoids circular imports in NeMo)
spec = importlib.util.spec_from_file_location(
    "trainer_utils", "NeMo/nemo/utils/trainer_utils.py"
)
trainer_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer_utils)
resolve_trainer_cfg = trainer_utils.resolve_trainer_cfg


@hydra_runner(config_path="../conf/citrinet/", config_name="config_bpe")
def main(cfg):
    logging.info(f"Hydra config:\\n{OmegaConf.to_yaml(cfg)}")

    trainer = pl.Trainer(**resolve_trainer_cfg(cfg.trainer))
    exp_manager(trainer, cfg.get("exp_manager", None))

    asr_model = EncDecCTCModelBPE(cfg=cfg.model, trainer=trainer)

    # Load weights from the Nepali Conformer checkpoint (proximal transfer)
    asr_model.maybe_init_from_pretrained_checkpoint(cfg)

    trainer.fit(asr_model)

    # Run test evaluation if a test manifest is provided
    if hasattr(cfg.model, "test_ds") and cfg.model.test_ds.manifest_filepath is not None:
        if asr_model.prepare_test(trainer):
            trainer.test(asr_model)


if __name__ == "__main__":
    main()
'''

with open(NEMO_SCRIPT_PATH, "w") as f:
    f.write(patched_script)

print(f"Patched training script written to: {NEMO_SCRIPT_PATH}")

---
## Step 7 — Launch Fine-Tuning

We call the patched NeMo script with Hydra overrides to:
- Load the **Nepali Conformer-CTC-BPE checkpoint** (`init_from_ptl_ckpt`) as the starting point for cross-lingual transfer
- Point to our **mono-converted Newari manifests**
- Use the **pre-trained BPE tokenizer**

Training logs and checkpoints stream to WandB .



In [ ]:
!python /kaggle/working/NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py \
    --config-path="/kaggle/working/" \
    --config-name="finetune_config" \
    ++init_from_ptl_ckpt="/kaggle/input/datasets/jennypoudel/newari-data/Conformer-CTC-BPE--val_wer0.2177-epoch49.ckpt" \
    model.train_ds.manifest_filepath="/kaggle/working/mono_nemo_train.json" \
    model.validation_ds.manifest_filepath="/kaggle/working/mono_nemo_val.json" \
    model.test_ds.manifest_filepath="/kaggle/working/mono_nemo_test.json" \
    model.tokenizer.dir="/kaggle/input/datasets/jennypoudel/newari-data/" \
    model.tokenizer.type="bpe"